# Trader Performance vs. Bitcoin Market Sentiment Analysis

## Executive Summary
This notebook explores the relationship between trader performance and market sentiment. It integrates historical trading data with the Bitcoin Fear & Greed Index to identify actionable patterns and build predictive models for trader profitability.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import sys
import os

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data_processing import load_data, clean_fear_greed, clean_historical, merge_datasets, get_profiling_summary
from src.features import run_feature_engineering
from src.eda import run_all_eda
from src.analysis import run_hypothesis_tests, segment_traders
from src.modeling import prepare_modeling_data, train_and_evaluate_models, plot_feature_importance

## Phase 1 & 2: Data Understanding and Cleaning
Load and clean the datasets.

In [2]:
fg_path = '../fear_greed_index.csv'
hist_path = '../historical_data.csv'

fg_df, hist_df = load_data(fg_path, hist_path)

print("Fear & Greed Profiling:", get_profiling_summary(fg_df, "Fear & Greed"))
print("Historical Data Profiling:", get_profiling_summary(hist_df, "Historical"))

fg_clean = clean_fear_greed(fg_df)
hist_clean = clean_historical(hist_df)

2026-06-08 23:45:51,828 - INFO - Loading sentiment data from ../fear_greed_index.csv
2026-06-08 23:45:51,839 - INFO - Loading historical trader data from ../historical_data.csv


Fear & Greed Profiling: {'Dataset': 'Fear & Greed', 'Shape': (2644, 4), 'Missing Values': 0, 'Duplicates': 0}


2026-06-08 23:45:52,905 - INFO - Cleaning sentiment data
2026-06-08 23:45:52,912 - INFO - Cleaning historical data


Historical Data Profiling: {'Dataset': 'Historical', 'Shape': (211224, 16), 'Missing Values': 0, 'Duplicates': 0}


## Phase 3 & 4: Feature Engineering and Dataset Integration
Merge the datasets based on trade dates and create sentiment and trader-specific features.

In [3]:
merged_df = merge_datasets(fg_clean, hist_clean)
print(f"Merged Dataset Shape: {merged_df.shape}")

final_df = run_feature_engineering(merged_df)
print(f"Final Dataset Shape (with features): {final_df.shape}")
final_df.head()

2026-06-08 23:45:53,979 - INFO - Merging datasets


Merged Dataset Shape: (211224, 20)
Final Dataset Shape (with features): (211224, 32)


,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,...,Sentiment_Rolling_30d,Sentiment_Momentum_7d,Is_Profitable,Trade_Value_Category,Total_Trades,Win_Rate,Average_PnL,Median_PnL,Total_Volume_USD,Average_Trade_Size
0,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,ETH,1897.9,0.08240,156.39,BUY,01-05-2023 01:06,0.0967,Open Long,0.0,...,63.0,0.0,0,Low,815,0.455215,-38.286626,0.0,1409902.00,1729.941104
1,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,ETH,1898.6,0.07220,137.08,BUY,01-05-2023 01:06,0.1791,Open Long,0.0,...,63.0,0.0,0,Low,815,0.455215,-38.286626,0.0,1409902.00,1729.941104
2,0x3998f134d6aaa2b6a5f723806d00fd2bbbbce891,ETH,1897.9,0.09670,183.53,BUY,01-05-2023 01:06,0.0000,Open Long,0.0,...,63.0,0.0,0,Low,815,0.455215,-38.286626,0.0,1409902.00,1729.941104
3,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,BTC,41866.0,0.58211,24370.62,SELL,05-12-2023 03:11,-0.0150,Open Short,0.0,...,69.0,0.0,0,Very High,14733,0.337134,145.481748,0.0,56543565.23,3837.885375
4,0xb1231a4a2dd02f2276fa3c5e2a2f3436e6bfed23,BTC,41867.0,0.01500,628.00,SELL,05-12-2023 03:11,0.0000,Open Short,0.0,...,69.0,0.0,0,High,14733,0.337134,145.481748,0.0,56543565.23,3837.885375


## Phase 5: Exploratory Data Analysis (EDA)
Generate visualizations to understand distributions and correlations.

In [4]:
run_all_eda(final_df, output_dir='../reports/figures')
print("Visualizations have been saved to the reports/figures directory.")

EDA visualizations saved to ../reports/figures
Visualizations have been saved to the reports/figures directory.


## Phase 6: Statistical Analysis
Test hypotheses regarding sentiment and trading performance.

In [5]:
stats_results = run_hypothesis_tests(final_df)
for test, res in stats_results.items():
    print(f"\n{test}:")
    for k, v in res.items():
        print(f"  {k}: {v}")


ANOVA_PnL_by_Sentiment:
  F-Statistic: 8.902023640443833
  p-value: 3.488463778558655e-07

Kruskal_PnL_by_Sentiment:
  H-Statistic: 1225.3290576886104
  p-value: 5.1417018042777e-264

ChiSquare_WinRate_Sentiment:
  Chi2: 821.200872939472
  p-value: 1.963233701803456e-176


## Phase 7: Trader Segmentation
Cluster traders based on their behaviors.

In [6]:
trader_segments = segment_traders(final_df, output_dir='../reports/figures')
trader_segments.groupby('Cluster').mean()

,Win_Rate,Average_PnL,Median_Size,Total_Trades,PCA1,PCA2
Cluster,,,,,,
0,0.441452,38.380504,492.628571,18559.428571,-1.402029,-0.212704
1,0.346208,365.725364,957.797000,1691.600000,1.919592,-0.464682
2,0.399487,22.694397,6544.545000,6211.000000,-0.032142,3.049656
3,0.404324,54.626797,930.240000,3357.111111,0.015585,-0.127054


## Phase 9: Predictive Modeling
Build machine learning models to predict if a trade will be profitable.

In [7]:
X, y = prepare_modeling_data(final_df)
model_results, fitted_models, feature_names = train_and_evaluate_models(X, y)

results_df = pd.DataFrame(model_results).T
print(results_df)

plot_feature_importance(fitted_models, feature_names, output_dir='../reports/figures')

                     Accuracy  Precision    Recall  F1 Score   ROC-AUC
Logistic Regression  0.767783   0.696508  0.771555  0.732114  0.684577
Random Forest        0.948964   0.931349  0.945608  0.938425  0.986056
XGBoost              0.942952   0.895037  0.975711  0.933634  0.985156
